# Example 3.7.3 / 1: Measure Violations Before Solving

Companion notebook for **3.7.3 Checking a Candidate Starting Solution** in the AMPL lab Documentation, section "Measure Violations Before Solving". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Measure Violations Before Solving

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= 0;                        # First decision, checked at a candidate value
var y >= 0;                        # Second decision, checked at a candidate value
subject to Capacity: x + y <= 10;  # Shared capacity of x and y
subject to Balance: x - y = 2;     # x must exceed y by exactly 2

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.var["x"].set_value(8)
ampl.var["y"].set_value(5)

In [5]:
# run.py
for name in ["Capacity", "Balance"]:
    c = ampl.con[name]
    print(name, "body", c.body(), "slack", c.slack())

Capacity body 13.0 slack -3.0
Balance body 3.0 slack -1.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
# Candidate values are assigned in model.run.

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
let x := 8;
let y := 5;
display Capacity.body, Capacity.slack;
display Balance.body, Balance.slack;

Capacity.body = 13
Capacity.slack = -3

Balance.body = 3
Balance.slack = -1

